### 1. Objective

Monthly Follow-up / Calling analysis from Athena:

- **Schedule Leads** — unique scheduled leads by Month × Source / Project
- **Mean Call Duration (Seconds)** — average of valid durations
- **Total Calls** — COUNT(DISTINCT call_id)

Excel: **3 sheets** — `By_Source`, `By_Project`, `Monthly_Summary`.

Fast path: configured DB + known CRM tables only (no multi-database scan).


### 2. Configuration


In [7]:
from __future__ import annotations

from datetime import date, timedelta
from pathlib import Path
from typing import Any

import awswrangler as wr
import boto3
import numpy as np
import pandas as pd
from IPython.display import display
from openpyxl import Workbook
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from openpyxl.utils.dataframe import dataframe_to_rows

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

ANALYSIS_START_DATE = date(2026, 1, 1)
ANALYSIS_END_DATE = date.today()

SCHEDULE_STAGE_OVERRIDE: str | None = None
SCHEDULE_STATUS_OVERRIDE: str | None = None

OUTPUT_XLSX = Path("/Users/lekhanigam/data/leads_sv_closure_reports/Followup_Calling_Monthly_Analysis.xlsx")
ENV_CANDIDATES = [Path("/Users/lekhanigam/data/.env"), Path(".env")]

START_TS = f"TIMESTAMP '{ANALYSIS_START_DATE.isoformat()} 00:00:00'"
END_EXCL = ANALYSIS_END_DATE + timedelta(days=1)
END_TS_EXCL = f"TIMESTAMP '{END_EXCL.isoformat()} 00:00:00'"

print("Window:", ANALYSIS_START_DATE, "→", ANALYSIS_END_DATE)
print("Output:", OUTPUT_XLSX)


Window: 2026-01-01 → 2026-09-08
Output: /Users/lekhanigam/data/Followup_Calling_Monthly_Analysis.xlsx


### 3. Athena Connection


In [8]:
def _aws_session(env: dict) -> "boto3.Session":
    """SSO profile — run: aws sso login --profile <AWS_PROFILE>"""
    profile = (env.get("AWS_PROFILE") or "asbl").strip()
    region = (env.get("AWS_REGION") or "ap-south-1").strip()
    session = boto3.Session(profile_name=profile, region_name=region)
    try:
        session.client("sts").get_caller_identity()
    except Exception as e:
        raise RuntimeError(
            f"AWS SSO login required. Run:\n  aws sso login --profile {profile}\n({e})"
        ) from e
    return session


def _load_env(path: Path) -> dict[str, str]:
    out: dict[str, str] = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        out[k.strip()] = v.strip()
    return out


_env: dict[str, str] = {}
for _p in ENV_CANDIDATES:
    if _p.exists():
        _env = _load_env(_p)
        print("Loaded", _p)
        break
else:
    raise FileNotFoundError("No .env found")

AWS_REGION = _env.get("AWS_REGION", "ap-south-1")
ATHENA_DATABASE = _env.get("ATHENA_DATABASE", "leadsdb-prod")
ATHENA_S3_OUTPUT = _env.get("ATHENA_OUTPUT_LOCATION", "s3://metabase-asbl/athena-query-results/")

# primary is the valid workgroup in this account (asbl_analytics_prod may not exist)
WORKGROUP_CANDIDATES = []
for wg in [_env.get("ATHENA_WORKGROUP", ""), "primary"]:
    if wg and wg not in WORKGROUP_CANDIDATES:
        WORKGROUP_CANDIDATES.append(wg)

AWS_PROFILE = _env.get("AWS_PROFILE", "asbl")
session = _aws_session(_env)
print("AWS profile:", AWS_PROFILE)
ident = session.client("sts").get_caller_identity()
print("AWS account:", ident.get("Account"))

ATHENA_WORKGROUP = WORKGROUP_CANDIDATES[0]


def athena_query(sql: str, database: str | None = None) -> pd.DataFrame:
    return wr.athena.read_sql_query(
        sql=sql,
        database=database or ATHENA_DATABASE,
        boto3_session=session,
        workgroup=ATHENA_WORKGROUP,
        s3_output=ATHENA_S3_OUTPUT,
        ctas_approach=False,
    )


# Probe with a real SELECT (SHOW TABLES via wrangler often returns empty)
_last_err = None
for wg in WORKGROUP_CANDIDATES:
    ATHENA_WORKGROUP = wg
    try:
        probe = athena_query(
            'SELECT 1 AS ok FROM leads LIMIT 1',
            database=ATHENA_DATABASE,
        )
        if probe is not None and not probe.empty:
            print("Athena OK | db:", ATHENA_DATABASE, "| workgroup:", ATHENA_WORKGROUP)
            break
        _last_err = f"empty probe via workgroup={wg}"
        print(_last_err)
    except Exception as e:
        _last_err = f"{type(e).__name__}: {e}"
        print("workgroup failed:", wg, "→", str(e)[:220])
else:
    raise RuntimeError(
        "Could not query Athena leads table. Check credentials/workgroup/database. "
        f"Last error: {_last_err}"
    )


Loaded /Users/lekhanigam/data/.env
AWS account: 737866084260
Athena OK | db: leadsdb-prod | workgroup: primary


### 4. Resolve Tables & Columns (fast)

Configured database only. Prefers `leads`, `leadcallrecords`, `leadstagehistories`, `leadsources`, `projects`.


In [9]:
def _norm(s: str) -> str:
    return "".join(ch for ch in str(s).lower() if ch.isalnum())


# Known CRM dump names. DESCRIBE/SHOW TABLES often return empty via wrangler —
# probe with SELECT LIMIT 1 instead.
PREFERRED = {
    "LEAD": ["leads"],
    "CALL": ["leadcallrecords", "lead_call_records", "calls"],
    "STAGE": ["leadstagehistories", "lead_stage_histories", "stagehistories"],
    "SOURCE": ["leadsources", "sources", "lead_sources"],
    "PROJECT": ["projects", "project"],
}


def probe_table(table: str) -> pd.DataFrame | None:
    """Return a 1-row sample if table is readable, else None."""
    for sql in (
        f'SELECT * FROM {table} LIMIT 1',
        f'SELECT * FROM "{ATHENA_DATABASE}"."{table}" LIMIT 1',
    ):
        try:
            df = athena_query(sql, database=ATHENA_DATABASE)
            if df is not None and len(df.columns) > 0:
                return df
        except Exception as e:
            last = f"{type(e).__name__}: {e}"
            continue
    return None


def pick_table(candidates: list[str]) -> tuple[str | None, pd.DataFrame | None]:
    for c in candidates:
        sample = probe_table(c)
        if sample is not None:
            return c, sample
    return None, None


LEAD_TABLE, lead_sample = pick_table(PREFERRED["LEAD"])
CALL_TABLE, call_sample = pick_table(PREFERRED["CALL"])
STAGE_TABLE, stage_sample = pick_table(PREFERRED["STAGE"])
SOURCE_TABLE, source_sample = pick_table(PREFERRED["SOURCE"])
PROJECT_TABLE, project_sample = pick_table(PREFERRED["PROJECT"])

print("LEAD   :", LEAD_TABLE, "| cols:", 0 if lead_sample is None else len(lead_sample.columns))
print("CALL   :", CALL_TABLE, "| cols:", 0 if call_sample is None else len(call_sample.columns))
print("STAGE  :", STAGE_TABLE, "| cols:", 0 if stage_sample is None else len(stage_sample.columns))
print("SOURCE :", SOURCE_TABLE)
print("PROJECT:", PROJECT_TABLE)

missing = [n for n, t in [("leads", LEAD_TABLE), ("calls", CALL_TABLE), ("stages", STAGE_TABLE)] if not t]
if missing:
    raise RuntimeError(
        f"Required tables not readable: {missing}. "
        "Connection works but SELECT * LIMIT 1 failed — check table names in Athena."
    )


def cols_from_sample(sample: pd.DataFrame | None) -> list[str]:
    return list(sample.columns) if sample is not None else []


def pick_col(cols: list[str], synonyms: list[str]) -> str | None:
    ncol = {_norm(c): c for c in cols}
    for s in synonyms:
        if _norm(s) in ncol:
            return ncol[_norm(s)]
    for s in synonyms:
        soft = sorted([c for c in cols if _norm(s) in _norm(c)], key=len)
        if soft:
            return soft[0]
    return None


lead_cols = cols_from_sample(lead_sample)
call_cols = cols_from_sample(call_sample)
stage_cols = cols_from_sample(stage_sample)
source_cols = cols_from_sample(source_sample)
project_cols = cols_from_sample(project_sample)

COL = {
    "lead_pk": pick_col(lead_cols, ["_id", "id"]),
    "lead_id": pick_col(lead_cols, ["leadid", "lead_id"]),
    "master_lead_id": pick_col(lead_cols, ["masterleadid", "master_lead_id"]),
    "lead_source": pick_col(lead_cols, ["source", "bornsource"]),
    "lead_project": pick_col(lead_cols, ["project"]),
    "call_pk": pick_col(call_cols, ["_id", "id"]),
    "call_id": pick_col(call_cols, ["callid", "call_id"]),
    "call_lead": pick_col(call_cols, ["lead", "leadid"]),
    "call_time": pick_col(call_cols, ["calltime", "call_time", "createdat"]),
    "call_duration": pick_col(call_cols, ["callduration", "call_duration", "duration"]),
    "call_dir": pick_col(call_cols, ["direction"]),
    "call_status": pick_col(call_cols, ["callstatus", "call_status", "status"]),
    "stage_pk": pick_col(stage_cols, ["_id", "id"]),
    "stage_lead": pick_col(stage_cols, ["lead", "leadid"]),
    "stage_name": pick_col(stage_cols, ["stage"]),
    "stage_status": pick_col(stage_cols, ["status"]),
    "stage_created": pick_col(stage_cols, ["createdat", "created_at", "updatedat"]),
    "source_pk": pick_col(source_cols, ["_id", "id"]) if source_cols else None,
    "source_name": pick_col(source_cols, ["name", "sourcename"]) if source_cols else None,
    "project_pk": pick_col(project_cols, ["_id", "id"]) if project_cols else None,
    "project_name": pick_col(project_cols, ["name"]) if project_cols else None,
}

print("Resolved columns:")
for k, v in COL.items():
    print(f"  {k:16s} → {v}")

need = ["lead_pk", "lead_id", "call_pk", "call_lead", "call_time", "stage_lead", "stage_name", "stage_created"]
missing_cols = [k for k in need if not COL[k]]
if missing_cols:
    raise RuntimeError(f"Missing critical columns: {missing_cols}")


def fq(table: str) -> str:
    # Unquoted table name in configured database (matches working psv notebook style)
    return table


LEAD   : leads | cols: 140
CALL   : leadcallrecords | cols: 60
STAGE  : leadstagehistories | cols: 64
SOURCE : leadsources
PROJECT: projects
Resolved columns:
  lead_pk          → _id
  lead_id          → leadid
  master_lead_id   → masterleadid
  lead_source      → source
  lead_project     → project
  call_pk          → _id
  call_id          → callid
  call_lead        → lead
  call_time        → calltime
  call_duration    → callduration
  call_dir         → direction
  call_status      → callstatus
  stage_pk         → _id
  stage_lead       → lead
  stage_name       → stage
  stage_status     → status
  stage_created    → createdat
  source_pk        → _id
  source_name      → name
  project_pk       → _id
  project_name     → name


### 5. Detect Schedule Stage / Call Direction / Status


In [10]:
# Distinct stages
stage_values = athena_query(f"""
SELECT CAST(h."{COL['stage_name']}" AS varchar) AS stage, COUNT(*) AS n
FROM {fq(STAGE_TABLE)} h
GROUP BY 1 ORDER BY n DESC
""")
display(stage_values.head(30))

status_values = pd.DataFrame(columns=["status", "n"])
if COL["stage_status"]:
    status_values = athena_query(f"""
    SELECT CAST(h."{COL['stage_status']}" AS varchar) AS status, COUNT(*) AS n
    FROM {fq(STAGE_TABLE)} h
    GROUP BY 1 ORDER BY n DESC
    LIMIT 50
    """)
    display(status_values.head(20))


def _score_schedule(label: str) -> int:
    s = str(label).strip().lower()
    if not s or s in {"nan", "none"}:
        return -1
    score = 0
    if "schedul" in s:
        score += 5
    if "site visit" in s or "sitevisit" in s.replace(" ", ""):
        score += 3
    if any(x in s for x in ["post site", "visited", "closure", "blocked", "initiated", "completed"]):
        score -= 4
    return score


SCHEDULE_MODE, SCHEDULE_VALUE = None, None
if SCHEDULE_STAGE_OVERRIDE:
    SCHEDULE_MODE, SCHEDULE_VALUE = "stage", SCHEDULE_STAGE_OVERRIDE
elif SCHEDULE_STATUS_OVERRIDE:
    SCHEDULE_MODE, SCHEDULE_VALUE = "status", SCHEDULE_STATUS_OVERRIDE
else:
    cands = []
    for _, r in stage_values.iterrows():
        cands.append(("stage", str(r["stage"]), _score_schedule(r["stage"]), int(r["n"])))
    for _, r in status_values.iterrows():
        cands.append(("status", str(r["status"]), _score_schedule(r["status"]), int(r["n"])))
    cands.sort(key=lambda x: (x[2], x[3]), reverse=True)
    if cands and cands[0][2] > 0:
        SCHEDULE_MODE, SCHEDULE_VALUE = cands[0][0], cands[0][1]

if not SCHEDULE_VALUE:
    raise RuntimeError(
        "Could not detect Site Visit Scheduled. Set SCHEDULE_STAGE_OVERRIDE or SCHEDULE_STATUS_OVERRIDE."
    )
print(f"Schedule definition: {SCHEDULE_MODE} = {SCHEDULE_VALUE!r}")

direction_values = pd.DataFrame(columns=["direction", "n"])
if COL["call_dir"]:
    direction_values = athena_query(f"""
    SELECT CAST(c."{COL['call_dir']}" AS varchar) AS direction, COUNT(*) AS n
    FROM {fq(CALL_TABLE)} c
    WHERE c."{COL['call_time']}" >= {START_TS} AND c."{COL['call_time']}" < {END_TS_EXCL}
    GROUP BY 1 ORDER BY n DESC
    """)
    display(direction_values)

status_values_c = pd.DataFrame(columns=["call_status", "n"])
if COL["call_status"]:
    status_values_c = athena_query(f"""
    SELECT CAST(c."{COL['call_status']}" AS varchar) AS call_status, COUNT(*) AS n
    FROM {fq(CALL_TABLE)} c
    WHERE c."{COL['call_time']}" >= {START_TS} AND c."{COL['call_time']}" < {END_TS_EXCL}
    GROUP BY 1 ORDER BY n DESC
    """)
    display(status_values_c)


def normalize_direction(raw) -> str:
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return "UNKNOWN"
    s = str(raw).strip().lower()
    if not s or s in {"nan", "none", "null"}:
        return "UNKNOWN"
    if s in {"inbound", "incoming", "in"} or "inbound" in s or "incoming" in s:
        return "INBOUND"
    if s in {"outbound", "outgoing", "out"} or "outbound" in s or "outgoing" in s:
        return "OUTBOUND"
    return "UNKNOWN"


def normalize_call_status(raw) -> str:
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return "UNKNOWN"
    s = str(raw).strip().lower().replace("_", "-").replace(" ", "-")
    if not s or s in {"nan", "none", "null"}:
        return "UNKNOWN"
    if "unanswer" in s or "no-answer" in s:
        return "MISSED"
    if s in {"answered", "answer", "completed", "connected"} or ("answer" in s and "unanswer" not in s):
        return "ANSWERED"
    if any(t in s for t in ["missed", "busy", "failed", "cancel", "no-answer", "unanswer"]):
        return "MISSED"
    if any(t in s for t in ["attempt", "dial", "ring", "initiat", "queue", "progress"]):
        return "ATTEMPTED"
    return "UNKNOWN"


DIR_MAP = {str(v): normalize_direction(v) for v in direction_values.get("direction", pd.Series(dtype=str))}
STATUS_MAP = {str(v): normalize_call_status(v) for v in status_values_c.get("call_status", pd.Series(dtype=str))}
print("Direction map:", DIR_MAP)
print("Status map:", STATUS_MAP)


,stage,n
0,New Lead,5988964
1,Lead Initiated,3404548
2,Not Interested,1969984
3,Pre Site Visit,462784
4,Post Site Visit,286952
5,Flat Blocked,9545
6,Sales Closure,7050


,status,n
0,NA,5953351
1,BACKLOG,2579882
2,SF,2311153
3,CF,1140664
4,FREEZED,109622
5,ONSITEVISIT,31186
6,REQUEST PENDING,3969


Schedule definition: stage = 'Pre Site Visit'


,direction,n
0,outbound,1795089
1,inbound,87108


,call_status,n
0,customer-missed,765292
1,agent-missed,629951
2,answered,432639
3,<NA>,38481
4,FAILED,10206
5,agent-busy,4968
6,UNKNOWN,405
7,dropped,255


Direction map: {'outbound': 'OUTBOUND', 'inbound': 'INBOUND'}
Status map: {'customer-missed': 'MISSED', 'agent-missed': 'MISSED', 'answered': 'ANSWERED', '<NA>': 'UNKNOWN', 'FAILED': 'MISSED', 'agent-busy': 'MISSED', 'UNKNOWN': 'UNKNOWN', 'dropped': 'UNKNOWN'}


### 6. Pull Calls + Schedule + Lead Dimensions


In [11]:
call_id_sql = (
    f'NULLIF(TRIM(CAST(c."{COL["call_id"]}" AS varchar)), \'\')' if COL["call_id"]
    else "CAST(NULL AS varchar)"
)
call_pk_sql = f'CAST(c."{COL["call_pk"]}" AS varchar)'
call_uid_sql = f"COALESCE({call_id_sql}, {call_pk_sql})"
dir_sql = f'CAST(c."{COL["call_dir"]}" AS varchar)' if COL["call_dir"] else "CAST(NULL AS varchar)"
st_sql = f'CAST(c."{COL["call_status"]}" AS varchar)' if COL["call_status"] else "CAST(NULL AS varchar)"
dur_sql = f'TRY_CAST(c."{COL["call_duration"]}" AS double)' if COL["call_duration"] else "CAST(NULL AS double)"

print("Fetching calls…")
calls_raw = athena_query(f"""
WITH base AS (
  SELECT
    {call_uid_sql} AS call_id,
    CAST(c."{COL['call_lead']}" AS varchar) AS lead_oid,
    c."{COL['call_time']}" AS call_timestamp,
    {dur_sql} AS call_duration_seconds,
    {dir_sql} AS raw_direction,
    {st_sql} AS raw_call_status,
    ROW_NUMBER() OVER (
      PARTITION BY {call_uid_sql}
      ORDER BY c."{COL['call_time']}" DESC, {call_pk_sql} DESC
    ) AS rn
  FROM {fq(CALL_TABLE)} c
  WHERE c."{COL['call_time']}" >= {START_TS}
    AND c."{COL['call_time']}" < {END_TS_EXCL}
)
SELECT call_id, lead_oid, call_timestamp, call_duration_seconds, raw_direction, raw_call_status
FROM base WHERE rn = 1
""")
print("Unique calls:", len(calls_raw))

sched_field = COL["stage_name"] if SCHEDULE_MODE == "stage" else COL["stage_status"]
sched_val_sql = str(SCHEDULE_VALUE).replace("'", "''")

print("Fetching schedule leads…")
sched_raw = athena_query(f"""
WITH ev AS (
  SELECT
    CAST(h."{COL['stage_lead']}" AS varchar) AS lead_oid,
    h."{COL['stage_created']}" AS schedule_timestamp,
    ROW_NUMBER() OVER (
      PARTITION BY CAST(h."{COL['stage_lead']}" AS varchar)
      ORDER BY h."{COL['stage_created']}" ASC, CAST(h."{COL['stage_pk']}" AS varchar) ASC
    ) AS rn
  FROM {fq(STAGE_TABLE)} h
  WHERE CAST(h."{sched_field}" AS varchar) = '{sched_val_sql}'
    AND h."{COL['stage_created']}" >= {START_TS}
    AND h."{COL['stage_created']}" < {END_TS_EXCL}
)
SELECT lead_oid, schedule_timestamp FROM ev WHERE rn = 1
""")
print("Unique scheduled leads:", len(sched_raw))

lead_oids = sorted(
    set(calls_raw["lead_oid"].dropna().astype(str)) | set(sched_raw["lead_oid"].dropna().astype(str))
)
print("Distinct leads to enrich:", len(lead_oids))


def chunks(seq, n=800):
    for i in range(0, len(seq), n):
        yield seq[i : i + n]


src_join = ""
src_name_expr = "CAST(NULL AS varchar)"
if SOURCE_TABLE and COL["lead_source"] and COL["source_pk"] and COL["source_name"]:
    src_join = (
        f'LEFT JOIN {fq(SOURCE_TABLE)} s '
        f'ON CAST(l."{COL["lead_source"]}" AS varchar) = CAST(s."{COL["source_pk"]}" AS varchar)'
    )
    src_name_expr = f'NULLIF(TRIM(CAST(s."{COL["source_name"]}" AS varchar)), \'\')'

proj_join = ""
proj_name_expr = "CAST(NULL AS varchar)"
if PROJECT_TABLE and COL["lead_project"] and COL["project_pk"] and COL["project_name"]:
    proj_join = (
        f'LEFT JOIN {fq(PROJECT_TABLE)} p '
        f'ON CAST(l."{COL["lead_project"]}" AS varchar) = CAST(p."{COL["project_pk"]}" AS varchar)'
    )
    proj_name_expr = f'NULLIF(TRIM(CAST(p."{COL["project_name"]}" AS varchar)), \'\')'

src_sel = (
    f'NULLIF(TRIM(CAST(l."{COL["lead_source"]}" AS varchar)), \'\')'
    if COL["lead_source"]
    else "CAST(NULL AS varchar)"
)
mlid_sel = (
    f'CAST(l."{COL["master_lead_id"]}" AS varchar)'
    if COL["master_lead_id"]
    else "CAST(NULL AS varchar)"
)

if not lead_oids:
    leads_dim = pd.DataFrame(
        columns=["lead_oid", "lead_id", "master_lead_id", "source_raw", "source_name", "project"]
    )
else:
    lead_parts = []
    for batch in chunks(lead_oids, 800):
        in_list = ",".join("'" + x.replace("'", "''") + "'" for x in batch)
        part = athena_query(f"""
        SELECT
          CAST(l."{COL['lead_pk']}" AS varchar) AS lead_oid,
          CAST(l."{COL['lead_id']}" AS varchar) AS lead_id,
          {mlid_sel} AS master_lead_id,
          {src_sel} AS source_raw,
          {src_name_expr} AS source_name,
          {proj_name_expr} AS project
        FROM {fq(LEAD_TABLE)} l
        {src_join}
        {proj_join}
        WHERE CAST(l."{COL['lead_pk']}" AS varchar) IN ({in_list})
        """)
        lead_parts.append(part)
    leads_dim = pd.concat(lead_parts, ignore_index=True).drop_duplicates("lead_oid")

print("Lead dim rows:", len(leads_dim))


def _blank_to_na(s: pd.Series) -> pd.Series:
    s = s.astype("string")
    return s.mask(s.str.strip().isin(["", "nan", "None", "<NA>", "null", "NULL"]))


for c in ["source_raw", "source_name", "project", "lead_id", "master_lead_id"]:
    if c in leads_dim.columns:
        leads_dim[c] = _blank_to_na(leads_dim[c])

leads_dim["report_source"] = leads_dim["source_name"].fillna(leads_dim["source_raw"]).fillna("UNKNOWN")
leads_dim["project"] = leads_dim["project"].fillna("UNKNOWN")


Fetching calls…
Unique calls: 1882098
Fetching schedule leads…
Unique scheduled leads: 12412
Distinct leads to enrich: 310887
Lead dim rows: 310827


### 7. Build Calling + Schedule Datasets


In [12]:
assert leads_dim.empty or leads_dim["lead_oid"].is_unique

calling_df = calls_raw.merge(leads_dim, on="lead_oid", how="left", validate="m:1")
calling_df["call_timestamp"] = pd.to_datetime(calling_df["call_timestamp"], errors="coerce")
calling_df["month"] = calling_df["call_timestamp"].dt.to_period("M").astype(str)
calling_df["direction"] = calling_df["raw_direction"].map(
    lambda x: DIR_MAP.get(str(x), normalize_direction(x))
)
calling_df["normalized_call_status"] = calling_df["raw_call_status"].map(
    lambda x: STATUS_MAP.get(str(x), normalize_call_status(x))
)
calling_df["call_duration_seconds"] = pd.to_numeric(calling_df["call_duration_seconds"], errors="coerce")
calling_df["report_source"] = calling_df["report_source"].astype("string").fillna("UNKNOWN")
calling_df["project"] = calling_df["project"].astype("string").fillna("UNKNOWN")
calling_df = calling_df.drop_duplicates(subset=["call_id"], keep="first")

sched_df = sched_raw.merge(leads_dim, on="lead_oid", how="left", validate="m:1")
sched_df["schedule_timestamp"] = pd.to_datetime(sched_df["schedule_timestamp"], errors="coerce")
sched_df["month"] = sched_df["schedule_timestamp"].dt.to_period("M").astype(str)
sched_df["report_source"] = sched_df["report_source"].astype("string").fillna("UNKNOWN")
sched_df["project"] = sched_df["project"].astype("string").fillna("UNKNOWN")

print("calling_df:", len(calling_df), "| unique call_id:", calling_df["call_id"].nunique())
print("sched_df:", len(sched_df))
print("Sources sample:", sorted(calling_df["report_source"].dropna().unique())[:25])
display(calling_df.head(2))


calling_df: 1882098 | unique call_id: 1882098
sched_df: 12412
Sources sample: ['99-acres', 'AI Referral', 'ASBL DB', 'Aajneeti', 'BGD', 'CRM', 'Call', 'Channel Partner', 'Chatbot', 'Customer Referral', 'Event', 'Executive Referral', 'FIM Forms', 'GoogleAdsForm', 'Homes', 'Lead Gen Portal', 'Lead Referral', 'LinkedIn', 'Listing Portals', 'NoBrokerHood', 'Phone', 'Pre-launch Web', 'Repp_tech', 'Self Generated Lead', 'Social Media']


,call_id,lead_oid,call_timestamp,call_duration_seconds,raw_direction,raw_call_status,lead_id,master_lead_id,source_raw,source_name,project,report_source,month,direction,normalized_call_status
0,Xchanged322ce0b-ba34-4f10-9efd-59b569be54c1,6a112d9044d141578a40e9d9,2026-06-18 08:06:41.965,0.0,outbound,customer-missed,1191926,930751,69d4979d60cfeeb29bcd904f,FIM Forms,LEGACY,FIM Forms,2026-06,OUTBOUND,MISSED
1,1781775590795,6a015f11f7f146ffde9c986b,2026-06-18 09:39:50.153,0.0,outbound,customer-missed,1176643,920009,69d4979d60cfeeb29bcd9045,ASBL DB,LEGACY,ASBL DB,2026-06,OUTBOUND,MISSED


### 8. Monthly Pivots (Source & Project)


In [13]:
def month_label(period_str: str) -> str:
    try:
        return pd.Period(period_str, freq="M").strftime("%B %Y")
    except Exception:
        return str(period_str)


def month_dim_pivot(df, dim_col, value_col, agg, fill=0) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame(columns=["Month"])
    tmp = df.dropna(subset=["month"]).copy()
    tmp[dim_col] = tmp[dim_col].astype("string").fillna("UNKNOWN")
    if agg == "nunique":
        g = tmp.groupby(["month", dim_col], dropna=False)[value_col].nunique().reset_index(name="value")
    elif agg == "mean":
        g = tmp.groupby(["month", dim_col], dropna=False)[value_col].mean().reset_index(name="value")
    elif agg == "count_leads":
        g = tmp.groupby(["month", dim_col], dropna=False)["lead_oid"].nunique().reset_index(name="value")
    else:
        raise ValueError(agg)
    wide = g.pivot(index="month", columns=dim_col, values="value")
    if fill is not None:
        wide = wide.fillna(fill)
    wide = wide.sort_index()
    cols = sorted([c for c in wide.columns if str(c) != "UNKNOWN"], key=lambda x: str(x).lower())
    if any(str(c) == "UNKNOWN" for c in wide.columns):
        cols.append("UNKNOWN")
    wide = wide.reindex(columns=cols).reset_index().rename(columns={"month": "Month"})
    wide["Month"] = wide["Month"].map(month_label)
    wide.columns.name = None
    return wide


def as_int_pivot(wide: pd.DataFrame) -> pd.DataFrame:
    out = wide.copy()
    for c in out.columns:
        if c != "Month":
            out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0).round(0).astype(int)
    return out


def as_float_pivot(wide: pd.DataFrame, ndigits=2) -> pd.DataFrame:
    out = wide.copy()
    for c in out.columns:
        if c != "Month":
            out[c] = pd.to_numeric(out[c], errors="coerce").round(ndigits)
    return out


schedule_leads_pivot = as_int_pivot(month_dim_pivot(sched_df, "report_source", None, "count_leads", fill=0))
schedule_leads_by_project = as_int_pivot(month_dim_pivot(sched_df, "project", None, "count_leads", fill=0))

dur_ok = calling_df["call_duration_seconds"].notna() & (calling_df["call_duration_seconds"] >= 0)
duration_base = calling_df.loc[dur_ok]
mean_duration_pivot = as_float_pivot(
    month_dim_pivot(duration_base, "report_source", "call_duration_seconds", "mean", fill=np.nan)
)
mean_duration_by_project = as_float_pivot(
    month_dim_pivot(duration_base, "project", "call_duration_seconds", "mean", fill=np.nan)
)

total_calls_pivot = as_int_pivot(month_dim_pivot(calling_df, "report_source", "call_id", "nunique", fill=0))
total_calls_by_project = as_int_pivot(month_dim_pivot(calling_df, "project", "call_id", "nunique", fill=0))

print("Schedule Leads — Source")
display(schedule_leads_pivot)
print("Mean Duration — Source")
display(mean_duration_pivot)
print("Total Calls — Source")
display(total_calls_pivot)

_src_calls = int(total_calls_pivot.drop(columns=["Month"], errors="ignore").to_numpy().sum() or 0)
_proj_calls = int(total_calls_by_project.drop(columns=["Month"], errors="ignore").to_numpy().sum() or 0)
_src_sched = int(schedule_leads_pivot.drop(columns=["Month"], errors="ignore").to_numpy().sum() or 0)
_proj_sched = int(schedule_leads_by_project.drop(columns=["Month"], errors="ignore").to_numpy().sum() or 0)

assert _src_calls == len(calling_df), f"Source Total Calls {_src_calls} != {len(calling_df)}"
assert _proj_calls == len(calling_df), f"Project Total Calls {_proj_calls} != {len(calling_df)}"
assert _src_sched == len(sched_df), f"Source Schedule {_src_sched} != {len(sched_df)}"
assert _proj_sched == len(sched_df), f"Project Schedule {_proj_sched} != {len(sched_df)}"
print("Pivot accuracy checks PASSED")


Schedule Leads — Source


,Month,99-acres,Aajneeti,adonmo,AI Referral,ASBL DB,aurum,aurum-analytica-sv,BGD,builders-lead,Call,Channel Partner,Chatbot,Customer Referral,Event,Executive Referral,FIM Forms,GoogleAdsForm,grid-s,Homes,housing.com,Lead Gen Portal,Lead Referral,LinkedIn,magic-bricks,magic-bricks-vl,my-gate,NoBrokerHood,Phone,Pre-launch Web,repp-tech-sv,Repp_tech,Self Generated Lead,Social Media,US Event,Walk In,Web Lead,Whatsapp,Whatsapp FIMAds,Whatsapp GoogleAds,UNKNOWN
0,January 2026,96,0,2,0,375,32,0,17,20,0,6,0,21,21,220,109,3,5,0,20,0,23,0,36,6,46,0,111,0,6,12,44,7,10,1,714,192,2,0,0
1,February 2026,89,0,0,0,253,34,0,34,12,1,1,0,6,92,188,60,3,0,0,7,0,11,0,14,2,26,3,44,3,4,11,18,5,1,0,415,126,0,0,0
2,March 2026,95,1,0,0,133,34,1,45,4,1,1,0,6,24,188,83,2,3,0,8,0,12,1,6,2,52,1,57,1,8,12,27,4,2,0,372,68,3,2,0
3,April 2026,64,0,1,0,247,24,0,38,2,1,14,2,25,25,167,446,8,0,0,5,0,21,0,6,0,47,1,63,253,3,7,56,2,4,0,287,443,1,1,0
4,May 2026,61,0,0,0,197,25,1,18,6,0,4,4,11,35,134,196,5,0,2,5,0,25,0,3,0,59,2,37,274,0,6,46,1,4,0,244,337,1,1,0
5,June 2026,41,0,0,0,111,7,0,37,4,0,60,7,8,19,122,132,3,0,0,14,1,13,0,1,1,9,2,26,255,2,2,25,1,0,0,161,70,0,1,0
6,July 2026,68,0,2,0,73,24,0,53,6,1,94,3,9,24,131,113,6,1,0,6,0,15,0,4,2,6,1,28,213,0,2,19,3,1,0,174,50,0,1,0
7,August 2026,88,0,0,4,51,17,0,30,9,0,92,8,9,31,158,62,9,1,0,28,0,10,0,4,0,13,0,19,75,0,4,33,2,1,0,170,67,0,1,0
8,September 2026,14,0,0,1,10,0,0,8,1,0,12,1,0,36,59,17,3,0,0,2,0,4,0,1,0,4,1,16,3,0,1,7,0,2,0,42,11,0,0,5


Mean Duration — Source


,Month,99-acres,Aajneeti,adonmo,AI Referral,ASBL DB,aurum,aurum-analytica-sv,BGD,builders-lead,Call,Channel Partner,Chatbot,CRM,Customer Referral,estate-dekho,Event,Executive Referral,FIM Forms,GoogleAdsForm,grid-s,Homes,housing.com,Lead Gen Portal,Lead Referral,LinkedIn,Listing Portals,magic-bricks,magic-bricks-sv,magic-bricks-vl,my-gate,NoBrokerHood,Phone,Pre-launch Web,repp-tech-sv,Repp_tech,roofandfloor,Self Generated Lead,Social Media,US Event,Walk In,Web Lead,Whatsapp,Whatsapp FIMAds,Whatsapp GoogleAds,UNKNOWN
0,January 2026,9.85,7.50,10.92,NaN,12.43,13.96,NaN,11.31,18.88,12.02,38.53,NaN,7.19,19.31,16.0,11.59,16.80,13.01,14.21,14.34,NaN,11.66,NaN,14.18,15.00,NaN,16.31,NaN,12.92,12.06,9.46,23.16,9.71,15.71,9.02,NaN,17.51,17.65,36.00,20.86,13.63,11.92,11.83,11.11,NaN
1,February 2026,9.73,21.39,NaN,NaN,12.73,12.30,NaN,13.95,11.38,30.52,23.56,NaN,NaN,12.94,NaN,8.61,15.13,9.21,9.27,8.74,NaN,9.00,NaN,11.26,19.24,12.18,15.32,NaN,8.66,10.52,10.24,16.00,6.54,13.95,10.71,NaN,13.75,10.69,11.30,16.29,12.49,10.53,5.77,16.89,NaN
2,March 2026,8.54,23.02,NaN,NaN,11.86,8.68,11.03,12.09,8.96,9.18,33.84,NaN,NaN,21.01,NaN,5.52,11.76,8.14,15.09,42.75,NaN,7.48,NaN,9.51,15.39,NaN,9.62,NaN,11.60,9.37,10.10,21.98,8.76,12.03,8.85,NaN,10.32,10.74,13.68,18.53,10.92,9.05,8.85,15.26,NaN
3,April 2026,7.34,11.31,0.00,NaN,14.16,10.79,NaN,12.28,10.17,0.00,25.44,19.67,22.50,20.71,NaN,5.25,13.30,16.93,13.62,10.44,NaN,7.87,NaN,13.64,8.00,NaN,10.70,31.23,15.40,8.49,5.18,17.93,29.96,12.10,7.66,NaN,18.82,6.50,20.94,15.72,9.43,12.71,5.48,17.60,NaN
4,May 2026,6.99,12.72,0.00,NaN,11.05,8.32,NaN,8.97,8.68,7.80,47.22,14.73,15.00,22.83,0.0,6.98,11.26,10.11,14.61,12.09,16.32,6.05,NaN,16.42,NaN,NaN,13.42,0.00,10.58,6.84,8.87,13.78,17.83,2.56,6.03,8.00,10.35,12.48,12.44,16.23,8.23,8.98,8.03,9.46,NaN
5,June 2026,8.46,16.80,8.00,NaN,9.11,5.97,0.00,13.72,9.55,0.00,12.57,13.10,NaN,11.04,NaN,6.25,10.02,9.48,12.26,NaN,12.97,11.48,19.88,11.14,0.00,NaN,11.29,NaN,3.94,6.68,6.90,12.42,15.83,3.83,3.36,34.84,13.48,6.71,8.75,18.06,8.62,6.66,5.90,16.43,NaN
6,July 2026,8.70,52.00,6.45,NaN,8.87,13.14,NaN,9.94,11.11,26.53,9.03,15.20,36.50,14.04,NaN,6.42,10.98,9.56,16.88,0.00,9.08,12.82,17.67,10.51,0.00,NaN,17.69,NaN,8.17,10.48,14.08,11.90,11.89,1.40,6.13,NaN,10.38,14.60,6.42,14.91,9.20,7.98,0.50,15.12,NaN
7,August 2026,7.74,NaN,0.00,NaN,10.32,8.93,NaN,6.79,8.90,12.67,8.24,16.57,60.00,24.68,NaN,7.24,9.49,8.51,11.06,0.00,9.91,4.40,14.44,11.41,NaN,NaN,6.58,NaN,0.89,8.27,11.62,11.69,9.54,NaN,5.63,NaN,9.25,9.73,7.78,17.63,7.77,7.07,10.43,6.76,NaN
8,September 2026,6.44,NaN,NaN,0.0,6.89,8.43,NaN,10.26,10.49,NaN,7.94,4.10,NaN,33.69,NaN,7.43,9.12,7.25,12.38,0.00,5.80,6.40,8.44,12.72,0.00,NaN,3.10,NaN,NaN,5.59,19.73,16.25,7.10,NaN,2.95,NaN,7.16,12.69,0.00,16.44,8.43,7.18,5.90,0.71,18.62


Total Calls — Source


,Month,99-acres,Aajneeti,adonmo,AI Referral,ASBL DB,aurum,aurum-analytica-sv,BGD,builders-lead,Call,Channel Partner,Chatbot,CRM,Customer Referral,estate-dekho,Event,Executive Referral,FIM Forms,GoogleAdsForm,grid-s,Homes,housing.com,Lead Gen Portal,Lead Referral,LinkedIn,Listing Portals,magic-bricks,magic-bricks-sv,magic-bricks-vl,my-gate,NoBrokerHood,Phone,Pre-launch Web,repp-tech-sv,Repp_tech,roofandfloor,Self Generated Lead,Social Media,US Event,Walk In,Web Lead,Whatsapp,Whatsapp FIMAds,Whatsapp GoogleAds,UNKNOWN
0,January 2026,29996,4,13,0,21412,1830,0,1156,966,68,126,0,1,663,1,1794,6714,12577,428,52,0,2070,0,762,6,0,1489,0,201,6739,162,7529,133,111,1382,0,938,335,155,1848,53730,16251,157,163,0
1,February 2026,26377,10,1,0,13093,4511,0,3076,998,31,60,0,0,270,0,87805,6508,14037,1221,90,0,1927,0,612,5,11,851,0,196,6502,201,5875,153,320,2252,0,742,442,90,1541,45919,13236,67,101,0
2,March 2026,45214,23,0,0,10243,7579,31,8705,697,21,93,0,0,189,0,33087,6968,22933,697,30,0,1957,0,416,32,0,594,0,95,18494,508,6329,65,592,3883,0,756,199,76,1162,73033,12530,2060,350,0
3,April 2026,36404,15,7,0,14349,3641,0,5649,669,6,271,195,2,679,0,11800,5167,34080,1015,28,0,1972,0,658,12,0,269,8,113,17700,273,6428,10742,280,3290,0,1733,98,72,1255,60130,79977,1232,246,0
4,May 2026,39822,5,3,0,20038,3850,0,5733,1121,20,138,338,1,667,9,6869,5384,30879,650,10,774,972,0,961,0,0,400,10,23,29169,473,4951,26665,16,2296,73,1229,91,49,1649,48791,90041,550,221,0
5,June 2026,25908,5,4,0,12280,1551,13,5880,975,2,4649,438,0,394,0,30546,4363,21092,402,0,367,817,227,1020,2,0,239,0,21,4289,60,3698,35039,21,735,5,694,176,49,1319,28039,31214,106,47,0
6,July 2026,33371,2,3,0,6428,2426,0,8483,817,11,10080,130,6,453,0,17228,4099,17742,649,4,271,770,542,730,11,0,214,0,24,879,75,3272,27228,11,352,0,797,166,63,1525,28224,10996,12,83,0
7,August 2026,20758,0,3,0,5136,1622,0,6324,546,14,8106,95,1,366,0,12940,4913,12064,1538,25,77,756,591,674,0,0,166,0,19,2323,35,3258,14791,0,342,0,1095,131,42,1871,34350,12622,169,113,0
8,September 2026,3767,0,0,3,1029,179,0,787,69,0,1515,21,0,107,0,19756,1432,2402,399,1,15,221,162,92,2,0,72,0,1,1191,11,834,1352,0,79,0,185,54,21,437,4455,2372,97,17,82


Pivot accuracy checks PASSED


### 9. Monthly Summary + Excel (3 sheets)


In [14]:
def summarize_calls(df: pd.DataFrame) -> dict:
    n = len(df)
    answered = int((df["normalized_call_status"] == "ANSWERED").sum())
    missed = int((df["normalized_call_status"] == "MISSED").sum())
    valid = df.loc[
        df["call_duration_seconds"].notna() & (df["call_duration_seconds"] >= 0),
        "call_duration_seconds",
    ]
    return {
        "Total Calls": n,
        "Inbound Calls": int((df["direction"] == "INBOUND").sum()),
        "Outbound Calls": int((df["direction"] == "OUTBOUND").sum()),
        "Answered": answered,
        "Missed": missed,
        "Inbound Answered": int(
            ((df["direction"] == "INBOUND") & (df["normalized_call_status"] == "ANSWERED")).sum()
        ),
        "Inbound Missed": int(
            ((df["direction"] == "INBOUND") & (df["normalized_call_status"] == "MISSED")).sum()
        ),
        "Outbound Answered": int(
            ((df["direction"] == "OUTBOUND") & (df["normalized_call_status"] == "ANSWERED")).sum()
        ),
        "Outbound Missed": int(
            ((df["direction"] == "OUTBOUND") & (df["normalized_call_status"] == "MISSED")).sum()
        ),
        "Average Call Duration Seconds": round(float(valid.mean()), 2) if len(valid) else np.nan,
        "Answer Rate": float(answered / n) if n else np.nan,
        "Miss Rate": float(missed / n) if n else np.nan,
    }


monthly_rows = []
for month, g in calling_df.dropna(subset=["month"]).groupby("month"):
    row = {"Month": month_label(month)}
    row.update(summarize_calls(g))
    monthly_rows.append(row)
calling_summary = pd.DataFrame(monthly_rows).sort_values("Month").reset_index(drop=True)
display(calling_summary)

HEADER_FILL = PatternFill("solid", fgColor="1F4E79")
HEADER_FONT = Font(color="FFFFFF", bold=True)
TITLE_FONT = Font(bold=True, size=14, color="006600")
SECTION_FONT = Font(bold=True, size=12, color="006600")
THIN = Border(
    left=Side(style="thin", color="B0B0B0"),
    right=Side(style="thin", color="B0B0B0"),
    top=Side(style="thin", color="B0B0B0"),
    bottom=Side(style="thin", color="B0B0B0"),
)


def _auto_width(ws, max_width=22):
    for col_idx, col_cells in enumerate(ws.columns, start=1):
        vals = [len(str(c.value)) for c in col_cells[:80] if c.value is not None]
        width = min(max((max(vals) + 2) if vals else 12, 12), max_width)
        if col_idx == 1:
            width = max(width, 18)
        ws.column_dimensions[get_column_letter(col_idx)].width = width


def write_section_table(ws, title, df, start_row, is_float=False) -> int:
    ws.cell(start_row, 1, title).font = SECTION_FONT
    start_row += 1
    if df is None or df.empty:
        ws.cell(start_row, 1, "(no rows)")
        return start_row + 2
    header_row = start_row
    for r_i, row in enumerate(dataframe_to_rows(df, index=False, header=True), start=start_row):
        for c_i, val in enumerate(row, start=1):
            if isinstance(val, float) and np.isnan(val):
                val = None
            cell = ws.cell(r_i, c_i, val)
            cell.border = THIN
            cell.alignment = Alignment(
                vertical="center", horizontal="center" if c_i > 1 else "left"
            )
            if r_i == header_row:
                cell.fill = HEADER_FILL
                cell.font = HEADER_FONT
            elif c_i > 1 and isinstance(val, (int, float)):
                cell.number_format = "0.00" if is_float else "#,##0"
    if df.shape[1] > 1 and len(df) > 0:
        rng = f"B{header_row + 1}:{get_column_letter(df.shape[1])}{header_row + len(df)}"
        ws.conditional_formatting.add(
            rng,
            ColorScaleRule(
                start_type="min",
                start_color="FFFFFF",
                mid_type="percentile",
                mid_value=50,
                mid_color="C6EFCE",
                end_type="max",
                end_color="63BE7B",
            ),
        )
    _auto_width(ws)
    return header_row + len(df) + 3


def build_stacked_sheet(ws, title, blurb, schedule_df, mean_df, total_df):
    ws.cell(1, 1, title).font = TITLE_FONT
    ws.cell(2, 1, blurb)
    ws.cell(
        3,
        1,
        f"Period: {ANALYSIS_START_DATE} → {ANALYSIS_END_DATE} | Duration: Seconds | "
        f"Schedule: {SCHEDULE_MODE}={SCHEDULE_VALUE!r}",
    )
    row = 5
    row = write_section_table(ws, "Schedule Leads", schedule_df, row, False)
    row = write_section_table(ws, "Mean Call Duration (Seconds)", mean_df, row, True)
    write_section_table(ws, "Total Calls", total_df, row, False)


wb = Workbook()
ws1 = wb.active
ws1.title = "By_Source"
build_stacked_sheet(
    ws1,
    "Follow-up / Calling — by Source",
    "Month × Source (unique scheduled leads / mean duration / distinct calls).",
    schedule_leads_pivot,
    mean_duration_pivot,
    total_calls_pivot,
)

ws2 = wb.create_sheet("By_Project")
build_stacked_sheet(
    ws2,
    "Follow-up / Calling — by Project",
    "Month × Project (unique scheduled leads / mean duration / distinct calls).",
    schedule_leads_by_project,
    mean_duration_by_project,
    total_calls_by_project,
)

ws3 = wb.create_sheet("Monthly_Summary")
ws3.cell(1, 1, "Monthly Calling Summary").font = TITLE_FONT
ws3.cell(2, 1, "From calling-level detail (1 row = 1 unique call)")
r0 = 4
for r_i, row in enumerate(dataframe_to_rows(calling_summary, index=False, header=True), start=r0):
    for c_i, val in enumerate(row, start=1):
        if isinstance(val, float) and np.isnan(val):
            val = None
        cell = ws3.cell(r_i, c_i, val)
        cell.border = THIN
        if r_i == r0:
            cell.fill = HEADER_FILL
            cell.font = HEADER_FONT
        else:
            col = calling_summary.columns[c_i - 1]
            if col in {"Answer Rate", "Miss Rate"} and isinstance(val, (int, float)):
                cell.number_format = "0.00%"
            elif col == "Average Call Duration Seconds" and isinstance(val, (int, float)):
                cell.number_format = "0.00"
            elif isinstance(val, (int, float)):
                cell.number_format = "#,##0"
ws3.freeze_panes = "A5"
_auto_width(ws3, 28)

wb.save(OUTPUT_XLSX)
print("Wrote", OUTPUT_XLSX)
print("Sheets:", wb.sheetnames)
print("Calls:", len(calling_df), "| Scheduled leads:", len(sched_df))
assert calling_df["call_id"].is_unique
assert OUTPUT_XLSX.exists()
print("DONE")


,Month,Total Calls,Inbound Calls,Outbound Calls,Answered,Missed,Inbound Answered,Inbound Missed,Outbound Answered,Outbound Missed,Average Call Duration Seconds,Answer Rate,Miss Rate
0,April 2026,300465,12704,287761,66467,221816,4876,7364,61591,214452,12.10,0.221214,0.738242
1,August 2026,147876,8751,139125,32617,112663,3233,5370,29384,107293,8.38,0.220570,0.761875
2,February 2026,239131,11882,227249,64352,170672,4744,6769,59608,163903,10.55,0.269108,0.713718
3,January 2026,171962,9416,162546,48915,119631,3646,5493,45269,114138,13.20,0.284452,0.695683
4,July 2026,178177,7769,170408,37808,138890,2929,4682,34879,134208,9.54,0.212193,0.779506
5,June 2026,216686,9390,207296,44684,170078,3448,5550,41236,164528,9.64,0.206215,0.784905
6,March 2026,259641,12743,246898,59586,195452,4688,7786,54898,187666,9.65,0.229494,0.752778
7,May 2026,324941,12183,312758,68234,248359,5126,6583,63108,241776,9.58,0.209989,0.764320
8,September 2026,43219,2205,41014,9949,32789,900,1286,9049,31503,7.86,0.230200,0.758671


Wrote /Users/lekhanigam/data/Followup_Calling_Monthly_Analysis.xlsx
Sheets: ['By_Source', 'By_Project', 'Monthly_Summary']
Calls: 1882098 | Scheduled leads: 12412
DONE
